# Attitude agility — how it works

This is the explanation and reference for the `quicksat` agility budget. It covers what the config file holds, and how a slew time follows from two numbers. It also explains why the spacecraft's mass properties come as named cases, and where the tool stops.

It is not a tutorial, on purpose. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides, and `docs/` holds the explanation and the reference. It still runs, because an explanation that cannot run soon stops matching the code it describes. It runs against `docs/sizing/data/`, its own copy of the sample satellite's input files.

One premise holds throughout: **a sizing model needs to know whether a 40° slew fits, not how the wheels share the command.** The whole wheel geometry reduces to two numbers per axis: the momentum and the torque that the wheels can apply about it. The slew calculation uses only these two. There is no distribution matrix and no load per wheel.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.agility.budget import AgilityBudget, AgilityConfig, Axis
from quicksat.mass.budget import MassBudget
from quicksat.utils.mission import Mission

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")



In [2]:
DATA = Path("docs") / "sizing" / "data"
config = AgilityConfig.from_yaml_file(DATA / "agility_config.yaml")
mission = Mission.from_yaml_file(DATA / "mission.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

roll = AgilityBudget(config, mission, Axis.ROLL, "first_guess", mass_budget=mass_data)

## The config file

The file has three sections. Two things are left out on purpose.

| section | holds |
|---|---|
| `inertia_cases` | named sets of mass properties; pick one at the call |
| `wheels` | the pyramid, its nameplate figures, and how much of them policy allows |
| `settling_time` | what the platform needs after a slew before the payload can work |

**No mass.** The mass budget computes it, and a second copy here could come to disagree with it. The same reason keeps the altitude out of every config except `mission.yaml`. The mass comes from an attached `MassBudget`, read at the case's own propellant fraction.

**No target duration.** How long a slew may take is a requirement placed *on* a spacecraft, not a property *of* it. A routine acquisition and an emergency reslew ask different times of the same platform. So the target duration is passed as an argument.

In [3]:
print(Path("docs/sizing/data/agility_config.yaml").read_text())

# Attitude agility settings: the spacecraft cases that can be slewed, the wheels
# that slew them, and how long the platform takes to settle afterwards.
#
# No mass anywhere. The mass budget owns it, and a second copy here would drift --
# the same reason altitude lives only in mission.yaml. Orbit comes from mission.yaml,
# for ground track speed. The target duration for a manoeuvre is not here
# either: that is asked of a spacecraft, not a property of one.

inertia_cases:
  # One case is one spacecraft, one set of mass properties. The names are yours
  # -- nothing in the code matches on them. Each case is EITHER an envelope
  # estimate OR a stated inertia, never both, so there is never a question which
  # of the two produced a number.

  first_guess:               # inertia estimated from a uniform box
    body:
      x: 1.5 m               # along track
      y: 1.5 m               # cross track
      z: 2.0 m               # nadir
    appendage_factor:        # uplift for arrays,

## Inertia cases, and why the two shapes are kept apart

A case is one spacecraft, with one set of mass properties. It is **either** an envelope estimate **or** a stated inertia. An envelope estimate is a box with an appendage factor per axis. A stated inertia is given as a mass properties report or a CAD model gives it. A case is never both, and never neither. A model validator checks this on load, and its error names the keys it found. So a typo in `appendage_factor` is reported as a missing envelope, rather than as a case that matches no shape at all.

They are kept apart because of what happens when the spacecraft changes. An envelope inertia is *derived* from the mass, so it follows the mass budget automatically. A stated inertia is a fixed number that does not. Nothing can tell you when the two no longer agree. If one case mixed them, you could not say which of the two behaviours you had.

The user chooses the names. Nothing in the code depends on them, because `for_case` is a dictionary lookup. An unlisted name raises an error. A fallback case would instead give results for a spacecraft that nobody asked about, with no warning.

In [4]:
for name, case in config.inertia_cases.items():
    budget = AgilityBudget(config, mission, Axis.ROLL, name, mass_budget=mass_data)
    body, uplift = case.body, case.appendage_factor
    if body is not None and uplift is not None:
        shape = "envelope"
        detail = (
            f"box {body.x:.1f} x {body.y:.1f} x {body.z:.1f}, "
            f"factor {uplift.roll}, at {case.propellant:.0f} propellant"
        )
    else:
        shape, detail = "stated", "stated outright; no mass involved"
    print(f"{name:14s} {shape:9s} {budget.inertia:8.1f}   {detail}")

try:
    config.for_case("no_such_case")
except KeyError as error:
    print(f"\nunknown name -> {error}")

first_guess    envelope     262.3 m2 kg   box 1.5 m x 1.5 m x 2.0 m, factor 1.07, at 100 % propellant
measured_bol   stated       264.7 m2 kg   stated outright; no mass involved
measured_eol   stated       250.0 m2 kg   stated outright; no mass involved

unknown name -> "No inertia case named 'no_such_case'. Configured: first_guess, measured_bol, measured_eol"


### Only an envelope case needs a mass

The mass enters the slew calculation only through the inertia. The rate is momentum over inertia, and the acceleration is torque over inertia. Nothing else in the model uses the spacecraft at all. So a case that states its inertia needs no mass, no mass budget, and no propellant fraction. Such a case has no mass to report. An envelope case with no mass budget to read raises an error, rather than guessing.

In [5]:
stated = AgilityBudget(config, mission, Axis.ROLL, "measured_bol")
print(f"a stated case, with no mass budget at all:  {stated.inertia:.1f}")
print(f"  and it still slews:  40 deg in {stated.slew_time(40 * u.deg):.1f}")

orphan = AgilityBudget(config, mission, Axis.ROLL, "first_guess")
try:
    _ = orphan.inertia
except ValueError as error:
    print(f"\nan envelope case with nothing to weigh:\n  {error}")

a stated case, with no mass budget at all:  264.7 m2 kg
  and it still slews:  40 deg in 63.7 s

an envelope case with nothing to weigh:
  Case 'first_guess' estimates its inertia from an envelope, so it needs a mass. Attach a mass_budget, pass mass=, or use a case that states its inertia outright.


### The propellant fraction lives in the case

The propellant fraction only means something when a mass is *scaled*. So it belongs to envelope cases, and stated cases do not have it. This also keeps it out of the call signature. To choose a point in the mission, you choose a case.

The effect is real, and it helps. The tanks empty over a mission, so the spacecraft gets lighter. It turns faster, and reaches a larger angle in the same time. Agility is one of the few budgets that improves with age.

In [6]:
rows = []
for name in ("first_guess", "measured_bol", "measured_eol"):
    budget = AgilityBudget(config, mission, Axis.ROLL, name, mass_budget=mass_data)
    rows.append({
        "case": name,
        "inertia": budget.inertia.value,
        "max_rate": budget.max_rate().value,
        "slew_40_deg": budget.slew_time(40 * u.deg).value,
        "reach_in_113_s": budget.achievable_angle(113 * u.s).value,
    })
pd.DataFrame(rows)

,case,inertia,max_rate,slew_40_deg,reach_in_113_s
0,first_guess,262.273,0.737,63.186,61.960
1,measured_bol,264.700,0.730,63.689,61.392
2,measured_eol,250.000,0.773,60.645,65.001


## The wheel geometry reduces to a projection

Four wheels sit on a pyramid whose symmetry axis is along yaw. This mounting puts **roll and pitch both in the base plane**. The geometry is the same for both axes, and only the inertia differs. So one implementation serves both, and takes the axis as an argument.

Each wheel contributes `cos(elevation)` of its momentum and torque to the base plane, and `cos(45°)` of that to each axis in the plane. Multiplied by the number of wheels, this gives the capability about the axis:

$$H_{axis} = n \cdot H_{wheel} \cdot \cos(\epsilon)\cos(45°) \qquad T_{axis} = n \cdot T_{wheel} \cdot \cos(\epsilon)\cos(45°)$$

Yaw gets `sin(elevation)` instead, and is the weak axis with this mounting. No slew case here uses it. It is still reported, because its limits are what a yaw manoeuvre would have to stay within.

The **momentum use factor** and the **torque derating** are policy, not hardware. The momentum held back stores disturbances, and the torque held back keeps control authority during the slew. So the two sit in the config beside the wheels. They are not folded into the vendor's nameplate figures.

In [7]:
wheels = config.wheels
print(f"nameplate, per wheel   {wheels.momentum:.1f}   {wheels.torque:.2f}")
print(f"policy allows          {wheels.momentum_use_factor:.1f} of momentum, "
      f"{wheels.torque_derating:.0f} of torque")
print(f"so usable per wheel    {roll.usable_momentum_per_wheel:.3f}   "
      f"{roll.usable_torque_per_wheel:.3f}")
print()
print(f"projection             cos({wheels.elevation:.1f}) x cos(45 deg) = "
      f"{roll.projection.value:.4f}")
print(f"about an in-plane axis {roll.axis_momentum():.3f}   {roll.axis_torque():.4f}")
print(f"about yaw              {roll.yaw_momentum:.3f}   -- the weak axis")

nameplate, per wheel   4.0 N m s   0.20 N m
policy allows          33.3 % of momentum, 75 % of torque
so usable per wheel    1.332 N m s   0.150 N m

projection             cos(26.5 deg) x cos(45 deg) = 0.6328
about an in-plane axis 3.372 N m s   0.3797 N m
about yaw              2.377 N m s   -- the weak axis


### One failed wheel halves the axis

With one of the four wheels failed, three remain. Only **two of the three** can run at full torque if the net momentum in the base plane is to stay zero. So both the momentum and the torque about the axis halve.

The geometry and the projection do *not* change. This is the same pyramid with one wheel failed, not an orthogonal mounting of three wheels. The rate and the acceleration both halve. The crossover angle is `ω²/α`, so it halves too. A degraded spacecraft is slower at every angle, and becomes momentum limited at smaller angles.

In [8]:
print(f"{'':22s}{'nominal':>12s}{'one failed':>12s}")
for label, nominal, degraded in (
    ("axis momentum", roll.axis_momentum(), roll.axis_momentum(degraded=True)),
    ("axis torque", roll.axis_torque(), roll.axis_torque(degraded=True)),
    ("max rate", roll.max_rate(), roll.max_rate(degraded=True)),
    ("max acceleration", roll.max_acceleration(), roll.max_acceleration(degraded=True)),
    ("crossover angle", roll.crossover_angle(), roll.crossover_angle(degraded=True)),
):
    print(f"  {label:20s}{nominal.value:12.4f}{degraded.value:12.4f}")

                           nominal  one failed
  axis momentum             3.3716      1.6858
  axis torque               0.3797      0.1898
  max rate                  0.7366      0.3683
  max acceleration          0.0829      0.0415
  crossover angle           6.5407      3.2703


## Two regimes, and the angle where they swap

$$\omega_{max} = \frac{H_{axis}}{I} \qquad \alpha_{max} = \frac{T_{axis}}{I} \qquad \theta_{cross} = \frac{\omega_{max}^2}{\alpha_{max}}$$

Below the crossover, the wheels never saturate. The slew accelerates to the halfway point, and decelerates to the end. This is a **triangular** rate profile, and the slew is torque limited:

$$t = 2\sqrt{\theta / \alpha} \qquad \omega_{peak} = \sqrt{\theta\alpha}$$

Above it, the wheels reach their momentum limit before the midpoint, so the slew coasts. This is a **trapezoid**, and the slew is momentum limited:

$$t = \frac{\theta}{\omega_{max}} + \frac{\omega_{max}}{\alpha} \qquad \omega_{peak} = \omega_{max}$$

Which of the two limits a slew is a main output, not a detail. It tells you whether more torque or more momentum would shorten the slew. Below the crossover, more torque helps. Above it, more torque does not help, and only more momentum would.

The momentum used gives the same information as a number. It reaches 1 for every slew past the crossover, and that is what momentum limited *means*. Below the crossover, the rest is spare momentum, which a larger slew would use.

In [9]:
crossover = roll.crossover_angle()
angles = [crossover * 0.5, crossover * 0.99, crossover * 1.01, crossover * 5]
print(f"crossover at {crossover:.2f}\n")
print(f"{'angle':>10}{'profile':>14}{'time':>10}{'peak rate':>12}{'momentum':>11}")
for angle in angles:
    print(f"{angle.value:9.2f}{roll.profile(angle).value:>14}"
          f"{roll.slew_time(angle).value:9.1f}s"
          f"{roll.peak_rate(angle).value:11.4f}"
          f"{roll.momentum_used(angle).value:10.1%}")

crossover at 6.54 deg

     angle       profile      time   peak rate   momentum
     3.27    triangular     12.6s     0.5208     70.7%
     6.48    triangular     17.7s     0.7329     99.5%
     6.61   trapezoidal     17.8s     0.7366    100.0%
    32.70   trapezoidal     53.3s     0.7366    100.0%


## The target duration is an argument, not config

`settling_time` is in the config, because it is a property of the platform. After any slew, the ADCS needs that long to damp out the motion before the payload can work. It is applied once at the end, not between steps.

The time *allowed* is a different kind of value. One operation may ask the spacecraft for 40° in 113 seconds, and another for 40° in 90 seconds. Neither is more correct than the other. So `time_margin`, `achievable_angle` and `tabulated_agility` all take a `target_duration`, and the budget answers for the one you give.

`achievable_angle` is the inverse solve: the same calculation, run backwards. So it has to agree with `slew_time` exactly, and a test checks this.

In [10]:
angle = 40 * u.deg
for target in (113 * u.s, 90 * u.s, 70 * u.s):
    margin = roll.time_margin(angle, target)
    reach = roll.achievable_angle(target)
    verdict = "fits" if margin.value >= 0 else "does not fit"
    print(f"in {target:5.0f}:  {angle:.0f} needs {roll.total_time(angle):.1f}, "
          f"{margin.to('percent'):+6.1f} -- {verdict:12s} (can reach {reach:.1f} within the allocated duration)")

print("\nthe inverse solve agrees with the forward one:")
reach = roll.achievable_angle(113 * u.s)
print(f"  {reach:.3f} takes {roll.total_time(reach):.3f}, the target exactly")

in   113 s:  40 deg needs 83.2 s,  +35.8 % -- fits         (can reach 62.0 deg within the allocated duration)
in    90 s:  40 deg needs 83.2 s,   +8.2 % -- fits         (can reach 45.0 deg within the allocated duration)
in    70 s:  40 deg needs 83.2 s,  -15.9 % -- does not fit (can reach 30.3 deg within the allocated duration)

the inverse solve agrees with the forward one:
  61.960 deg takes 113.000 s, the target exactly


### The ground passed during a slew

The shared mission's ground track speed turns a slew time into a distance. It is the length of ground track that passes while the satellite slews, and none of it is imaged. A payload operator measures a slew this way. This is also why agility is a budget rather than a specification.

In [11]:
print(f"ground track speed at {mission.altitude:.0f}:  {mission.ground_track_speed:.4f}")
for angle in (20 * u.deg, 40 * u.deg, 90 * u.deg):
    duration = roll.total_time(angle)
    print(f"  {angle:5.0f} costs {duration:6.1f} and "
          f"{roll.ground_distance(duration):.0f} of ground track")

ground track speed at 500 km:  7.0592 km / s


     20 deg costs   56.0 s and 396 km of ground track


     40 deg costs   83.2 s and 587 km of ground track
     90 deg costs  151.1 s and 1066 km of ground track


## The document view

`tabulated_agility()` lays out the slew table for reading. The requirement check is optional, and depends on the target duration. Given one, the table gains a margin column and a verdict column, and marks in red the slews that do not fit. Without one, those two columns are left out rather than hidden, because there is nothing to check against.

This differs from the other reports on purpose. `comments` on the delta-V budget hides a column that always exists. Here, the columns cannot be computed at all until you give a target to check against.

In [12]:
roll.tabulated_agility(target_duration=113 * u.s)

Slew [deg],Limited by,Slew [s],With settling [s],Peak rate [deg/s],Momentum used,Time margin,
5,torque,15.5,35.5,0.6440,87.4%,+218.1%,PASS
10,momentum,22.5,42.5,0.7366,100.0%,+166.2%,PASS
15,momentum,29.2,49.2,0.7366,100.0%,+129.5%,PASS
20,momentum,36.0,56.0,0.7366,100.0%,+101.7%,PASS
40,momentum,63.2,83.2,0.7366,100.0%,+35.8%,PASS
45,momentum,70.0,90.0,0.7366,100.0%,+25.6%,PASS
60,momentum,90.3,110.3,0.7366,100.0%,+2.4%,PASS
90,momentum,131.1,151.1,0.7366,100.0%,-25.2%,FAILS


In [13]:
plain = roll.tabulated_agility()
print("without a target duration, the check is not computed at all:")
print(f"  columns: {list(plain.data.columns)}")

without a target duration, the check is not computed at all:
  columns: ['angle', 'profile', 'slew_time', 'total_time', 'peak_rate', 'momentum_used']


## Limitations

What the agility budget does not do, on purpose:

- **No distribution matrix.** The wheel geometry reduces to a projection factor. So nothing here gives the torque or momentum of each wheel, or checks that each wheel stays within its own limits. A real ADCS design needs that, and a sizing model does not.
- **Rest to rest, one axis at a time.** There are no coupled manoeuvres, no gyroscopic cross-terms, and no products of inertia. The inertia is a single number per axis, not a tensor. This model does not cover a slew about roll and pitch together.
- **The profile is ideal.** The torque reverses instantly at the midpoint, with no jerk limit and no actuator lag. Settling is a fixed allowance, not derived from the control loop.
- **No disturbance torques.** Gravity gradient, aerodynamic and solar torques are left out. The momentum use factor and the torque derating hold capability back *for* them. But the reserve is a policy figure, not a calculation.
- **One failed wheel, and only the halving.** The degraded case assumes the remaining geometry still balances at half capability. A different failure, or a mounting that does not, is not covered.
- **Nothing checks a stated inertia.** An envelope case follows the mass budget. A stated one is a fixed number, and nothing warns you if the spacecraft grows. That is the downside of allowing a measured figure at all.